In [1]:
import pandas as pd
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import json
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F

/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0730 17:06:28.056000 31875 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
language_classifier = pipeline(
    "text-classification",
    model = "papluca/xlm-roberta-base-language-detection"
)

def lang_result(text):
    results = language_classifier(
        text,
        truncation=True
    )
    return results[0]["label"]

theme_classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0",
    multi_label = True
)

def theme_result(text, theme_labels):
    return theme_classifier(
        text,
        candidate_labels=theme_labels,
        hypothesis_template="This post discusses {}.",
        multi_label=True
    )

model_name = "yangheng/deberta-v3-base-absa-v1.1"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()


def aspect_sentiment(text, aspect, batch_size=16, max_length=512, stride=64):
    encoded = tokenizer(
        text,
        aspect,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding=True,
        return_tensors="pt"
    )

    input_keys = ["input_ids", "attention_mask", "token_type_ids"]
    input_keys = [k for k in input_keys if k in encoded]

    all_probs = []

    with torch.inference_mode():
        n_chunks = encoded["input_ids"].shape[0]

        for start in range(0, n_chunks, batch_size):
            end = start + batch_size

            batch = {
                k: encoded[k][start:end].to(device)
                for k in input_keys
            }

            outputs = model(**batch)
            probs = F.softmax(outputs.logits, dim=-1)
            all_probs.append(probs)

    avg_probs = torch.cat(all_probs, dim=0).mean(dim=0).cpu()

    return {
        model.config.id2label[i]: float(avg_probs[i])
        for i in range(len(avg_probs))
    }

Device set to use mps:0
Device set to use mps:0


Using device: mps


/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [3]:
def remove_links(text):
    url_pattern = re.compile(r'[\[(]?(?:https?://|www\.)\S+[\])]?' )
    return url_pattern.sub('', text)

In [4]:
BASE_DIR = Path.cwd()
file_path = (
    BASE_DIR
    / "brightdata_social_exports"
    / "facebook_datacenters_posts.json"
)

with file_path.open("r", encoding="utf-8") as f:
    facebook_posts = json.load(f)

In [9]:
english_post_ids = []
a = 0

for fb_post in facebook_posts:
    unclean_text = fb_post["content"]
    text = remove_links(unclean_text)
    language = lang_result(text)
    pid = fb_post["post_id"]
    if language == 'en':
        english_post_ids.append(pid)
    a += 1
    print("Posts scanned:", a, end="\r")

KeyboardInterrupt: 

In [10]:
print(len(english_post_ids))

43


In [13]:
all_post_ids = []

env_post_ids = []
env_post_sentiments = []
env_post_sentiment_degrees = []

infr_post_ids = []
infr_post_sentiments = []
infr_post_sentiment_degrees = []

util_post_ids = []
util_post_sentiments = []
util_post_sentiment_degrees = []

housing_post_ids = []
housing_post_sentiments = []
housing_post_sentiment_degrees = []

econ_post_ids = []
econ_post_sentiments = []
econ_post_sentiment_degrees = []

life_qual_post_ids = []
life_qual_post_sentiments = []
life_qual_post_sentiment_degrees = []

aesth_post_ids = []
aesth_post_sentiments = []
aesth_post_sentiment_degrees = []

gov_post_ids = []
gov_post_sentiments = []
gov_post_sentiment_degrees = []

tech_post_ids = []
tech_post_sentiments = []
tech_post_sentiment_degrees = []

not_useful_post_ids = []

themes = ["visual impact of datacenters", "data centers infrastructure", "impact of data centers on home utility availability and cost", "impact of data centers on housing costs and property values", "impact of data centers on the economy and job market", "quality of life around data centers, noise, and light pollution", "environmental impact of data centers", "government decisions and policies relating to data centers", "technology performance and growth"]

matched_posts = 0
more_than_one_theme_posts = 0

a = 0

for fb_post in facebook_posts:

    post_id = fb_post["post_id"]
    if post_id not in english_post_ids:
        continue
    all_post_ids.append(post_id)
    unclean_text = fb_post["content"]
    text = remove_links(unclean_text)

    final_labels = []

    theme_scores = theme_result(
        text,
        themes
    )
    
    for i in range(len(theme_scores['labels'])):
        if theme_scores['scores'][i] > 0.5:
            final_labels.append(theme_scores['labels'][i])
    
    if len(final_labels) > 0:
        matched_posts += 1
    else:
        not_useful_post_ids.append(post_id)
    
    if len(final_labels) > 1:
        more_than_one_theme_posts += 1
    

    for label in final_labels:
        total_sentiment = aspect_sentiment(text, label)
        post_sentiment = max(total_sentiment, key=total_sentiment.get)
        post_degree = max(total_sentiment.values())
        if label == "visual impact of datacenters":
            aesth_post_ids.append(post_id)
            aesth_post_sentiments.append(post_sentiment)
            aesth_post_sentiment_degrees.append(post_degree)
        if label == "data centers infrastructure":
            infr_post_ids.append(post_id)
            infr_post_sentiments.append(post_sentiment)
            infr_post_sentiment_degrees.append(post_degree)
        if label == "impact of data centers on home utility availability and cost":
            util_post_ids.append(post_id)
            util_post_sentiments.append(post_sentiment)
            util_post_sentiment_degrees.append(post_degree)
        if label == "impact of data centers on housing costs and property values":
            housing_post_ids.append(post_id)
            housing_post_sentiments.append(post_sentiment)
            housing_post_sentiment_degrees.append(post_degree)
        if label == "impact of data centers on the economy and job market":
            econ_post_ids.append(post_id)
            econ_post_sentiments.append(post_sentiment)
            econ_post_sentiment_degrees.append(post_degree)
        if label == "quality of life around data centers, noise, and light pollution":
            life_qual_post_ids.append(post_id)
            life_qual_post_sentiments.append(post_sentiment)
            life_qual_post_sentiment_degrees.append(post_degree)
        if label == "environmental impact of data centers":
            env_post_ids.append(post_id)
            env_post_sentiments.append(post_sentiment)
            env_post_sentiment_degrees.append(post_degree)
        if label == "government decisions and policies relating to data centers":
            gov_post_ids.append(post_id)
            gov_post_sentiments.append(post_sentiment)
            gov_post_sentiment_degrees.append(post_degree)
        if label == "technology performance and growth":
            tech_post_ids.append(post_id)
            tech_post_sentiments.append(post_sentiment)
            tech_post_sentiment_degrees.append(post_degree)

        a += 1
        print("Posts scanned:", a, end="\r")

print("Total posts scanned:", len(all_post_ids))
print("Total posts with a theme:", matched_posts)
print("Found environmental posts:", len(env_post_ids))
print("Found infrastructure posts:", len(infr_post_ids))
print("Found utility posts:", len(util_post_ids))
print("Found housing posts:", len(housing_post_ids))
print("Found economic posts:", len(econ_post_ids))
print("Found life quality posts:", len(life_qual_post_ids))
print("Found aesthetic posts:", len(aesth_post_ids))
print("Found government posts:", len(gov_post_ids))
print("Found technological posts:", len(tech_post_ids))
print(not_useful_post_ids)

Total posts scanned: 43
Total posts with a theme: 32
Found environmental posts: 1
Found infrastructure posts: 25
Found utility posts: 0
Found housing posts: 0
Found economic posts: 0
Found life quality posts: 0
Found aesthetic posts: 0
Found government posts: 4
Found technological posts: 26
['1187839470053865', '3634471446608526', '952461816746294', '146967316838450', '1471527749640053', '617849510352476', '556750613771779', '10166200822710454', '1085409245239958', '1145354989245307', '549828020481464']


In [12]:
posts_by_id = {post["post_id"]: post for post in facebook_posts}
not_useful_links = []

for id in not_useful_post_ids:
    post = posts_by_id.get(id)
    not_useful_links.append(post["url"])
print(not_useful_links)

['https://www.facebook.com/elizabethguzmanva/posts/pfbid0r63cLoDHCg39MbFjnuSnjqzV6fpvyHUN1LuUGHAyWJibvC5cTUQySW2a6YPumB3bl', 'https://www.facebook.com/MindroverLLC/posts/pfbid021YAeHdq8HugGVeusbJc7ZJ7sG81pSxouQFehM2k3pBsUDXs2JyUwGJsgFn6R86s3l', 'https://www.facebook.com/photo/?fbid=3870790862932792&set=a.902232666455308', 'https://www.facebook.com/newsouthconstructionsupply/posts/pfbid0KYK98kCaWzdL7bzy9b74CZAqUzb8ME7rMj6n6BHbiHRzvE8FbXkStmMgFRPvazXhl', 'https://www.facebook.com/BloombergAsia/posts/pfbid02a5udJt6GGeDquCTR3DSvPVqYFFPcscBoPpq4iwhQ2Qr5Pfyj7ULNwkgpnMF8iVb9l', 'https://www.facebook.com/GlobalFinTechSeries/posts/pfbid02VHQ96wUPdnZcCy2hhBu8jwLyyZ4VkPfcybvoMDaMH2nDL8KpeJsvh14zP3yry83Dl', 'https://www.facebook.com/NetsyncNetworkSolutionsMEA/posts/pfbid02VvAdh1onJ3dWCrj56FnaZqREGfgaFJbTKgL3npCavSmA1yLspD1m7NgqYr4nRF6bl', 'https://www.facebook.com/NewAlbanyDataCenter/posts/pfbid0ut4B63Dux9JP1SGVPjYQNkbngEiSXh6DNCkHCK6Ye2qRAmVNVhtNabCRT4fzx25el', 'https://www.facebook.com/marvellou

In [11]:
theme_lists = [env_post_ids, infr_post_ids, housing_post_ids, econ_post_ids, life_qual_post_ids, aesth_post_ids, gov_post_ids, tech_post_ids]
posts = pd.DataFrame(columns=["ids", "text", "date", "likes", "number of comments", "number of shares", "page followers", "is page verified", "is paid partnership", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])
post_ids = []
datacenters_keywords = ["datacenter", "data center", "datacentre", "data centre"]

posts_by_id = {post["post_id"]: post for post in facebook_posts}

for theme in theme_lists:
    df1 = pd.DataFrame(columns=["ids", "text", "date", "likes", "number of comments", "number of shares", "page followers", "is page verified", "is paid partnership", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])
    post_ids = []
    post_texts = []
    post_dates = []
    post_likes = []
    post_comment_numbers = []
    post_share_numbers = []
    post_page_followers = []
    post_is_page_verified = []
    post_is_paid_partnership = []


    for pid in theme:
        post_ids.append(pid)
        post = posts_by_id.get(pid)
        post_texts.append(post["content"])
        post_dates.append(post["date_posted"])
        if post["page_likes"] is None:
            post_likes.append(0)
        else:
            post_likes.append(post["page_likes"])
        post_comment_numbers.append(post["num_comments"])
        post_share_numbers.append(post["num_shares"])
        if post["page_followers"] is None:
            post_page_followers.append(0)
        else:
            post_page_followers.append(post["page_followers"])
        post_is_page_verified.append(post["page_is_verified"])
        post_is_paid_partnership.append(post["is_sponsored"])

    df1["ids"] = post_ids
    df1["text"] = post_texts
    df1["date"] = post_dates
    df1["likes"] = post_likes
    df1["number of comments"] = post_comment_numbers
    df1["number of shares"] = post_share_numbers
    df1["page followers"] = post_page_followers
    df1["is page verified"] = post_is_page_verified
    df1["is paid partnership"] = post_is_paid_partnership

    
    for col in ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology"]:
        df1[col] = False

    for col in ["environment sentiment", "environment sentiment degree", "infrastructure sentiment", "infrastructure sentiment degree", "housing sentiment", "housing sentiment degree", "economy sentiment", "economy sentiment degree", "life quality sentiment", "life quality sentiment degree", "aesthetics sentiment", "aesthetics sentiment degree", "government sentiment", "government sentiment degree", "technology sentiment", "technology sentiment degree"]:
        df1[col] = None

    if theme == env_post_ids:
        df1["environment"] = True
        df1["environment sentiment"] = env_post_sentiments
        df1["environment sentiment degree"] = env_post_sentiment_degrees
    if theme == infr_post_ids:
        df1["infrastructure"] = True
        df1["infrastructure sentiment"] = infr_post_sentiments
        df1["infrastructure sentiment degree"] = infr_post_sentiment_degrees
    if theme == housing_post_ids:
        df1["housing"] = True
        df1["housing sentiment"] = housing_post_sentiments
        df1["housing sentiment degree"] = housing_post_sentiment_degrees
    if theme == econ_post_ids:
        df1["economy"] = True
        df1["economy sentiment"] = econ_post_sentiments
        df1["economy sentiment degree"] = econ_post_sentiment_degrees
    if theme == life_qual_post_ids:
        df1["life quality"] = True
        df1["life quality sentiment"] = life_qual_post_sentiments
        df1["life quality sentiment degree"] = life_qual_post_sentiment_degrees
    if theme == aesth_post_ids:
        df1["aesthetics"] = True
        df1["aesthetics sentiment"] = aesth_post_sentiments
        df1["aesthetics sentiment degree"] = aesth_post_sentiment_degrees
    if theme == gov_post_ids:
        df1["government"] = True
        df1["government sentiment"] = gov_post_sentiments
        df1["government sentiment degree"] = gov_post_sentiment_degrees
    if theme == tech_post_ids:
        df1["technology"] = True
        df1["technology sentiment"] = tech_post_sentiments
        df1["technology sentiment degree"] = tech_post_sentiment_degrees
    posts = pd.concat([posts, df1], ignore_index=True)

/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_75635/2068141810.py:88: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  posts = pd.concat([posts, df1], ignore_index=True)
/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_75635/2068141810.py:88: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  posts = pd.concat([posts, df1], ignore_index=True)
/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_75635/2068141810.py:88: FutureWarning: The behavior of DataFrame concatenation wi

In [12]:
posts = posts.astype({
    "ids": "string",
    "text": "string",
    "date": "string",
    "likes": "int64",
    "number of comments": "int64",
    "number of shares": "int64",
    "page followers": "int64",
    "is page verified": "bool",
    "is paid partnership": "bool"
})

In [13]:
grouping_cols = ["ids", "text", "date", "likes", "number of comments", "number of shares", "page followers", "is page verified", "is paid partnership"]

theme_cols = ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology"]
sent_cols  = ["environment sentiment", "environment sentiment degree", "infrastructure sentiment", "infrastructure sentiment degree", "housing sentiment", "housing sentiment degree", "economy sentiment", "economy sentiment degree", "life quality sentiment", "life quality sentiment degree", "aesthetics sentiment", "aesthetics sentiment degree", "government sentiment", "government sentiment degree", "technology sentiment", "technology sentiment degree"]

def first_non_null(s):
    return s.dropna().iloc[0] if s.notna().any() else np.nan

agg = {c: "max" for c in theme_cols}              # True if any True
agg.update({c: first_non_null for c in sent_cols}) # keep the real sentiment if present

posts = posts.groupby(grouping_cols, as_index=False, dropna=False).agg(agg)

In [14]:
len(posts)

8

In [15]:
print(len(posts))
print(matched_posts)

8
8


In [ ]:
posts.to_json('facebook_ABSA_entire_dataframe.json', orient='records', indent=4)

In [16]:
posts.head(30)

,ids,text,date,likes,number of comments,number of shares,page followers,is page verified,is paid partnership,environment,...,economy sentiment,economy sentiment degree,life quality sentiment,life quality sentiment degree,aesthetics sentiment,aesthetics sentiment degree,government sentiment,government sentiment degree,technology sentiment,technology sentiment degree
0,1097163855757616,🚀 GTC 2025: Redefining the Future Data Center ...,2025-03-11T14:00:08.000Z,5400,1,0,5900,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.869296
1,1145486004286829,Oracle to spend $40B on Nvidia chips for OpenA...,2025-05-29T11:15:07.000Z,0,0,0,130000,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.728661
2,1187839470053865,Back on the doors today! I had meaningful conv...,2025-07-08T23:26:04.000Z,0,0,0,5900,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.854346,NaN,NaN
3,1215774918502343,You know to protect your data center from hack...,2016-10-01T07:44:16.000Z,4400,0,0,4500,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.963598
4,1312080475614060,The Future of Data Centers 2019 is brought to ...,2019-05-29T13:45:44.000Z,9400,0,1,11000,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Neutral,0.663518
5,3870790862932792,#ITtraining #ittrainingcenter\n#servertraining...,2020-03-03T06:44:31.000Z,377,0,0,370,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6130264733678555,"The firm plans to expand TOK1, its flagship da...",2023-03-14T05:31:19.000Z,797000,0,1,836000,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.786645
7,878897414395023,Expand your data storage capacity with the Sea...,2024-10-18T15:12:18.000Z,0,0,0,27000,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.966020


In [17]:
print(posts.columns)

Index(['ids', 'text', 'date', 'likes', 'number of comments',
       'number of shares', 'page followers', 'is page verified',
       'is paid partnership', 'environment', 'infrastructure', 'housing',
       'economy', 'life quality', 'aesthetics', 'government', 'technology',
       'environment sentiment', 'environment sentiment degree',
       'infrastructure sentiment', 'infrastructure sentiment degree',
       'housing sentiment', 'housing sentiment degree', 'economy sentiment',
       'economy sentiment degree', 'life quality sentiment',
       'life quality sentiment degree', 'aesthetics sentiment',
       'aesthetics sentiment degree', 'government sentiment',
       'government sentiment degree', 'technology sentiment',
       'technology sentiment degree'],
      dtype='object')


In [18]:
# calculating average sentiment based on theme:
# Weigh all posts by their degree in the numerator and denominator, means that the average sentiment will just be +/- 1 if there are only positive or negative themes but other than that does a pretty good job of weighing neutrality
def avg_sentiment_calculation(theme):
    theme_posts = posts[posts[theme] == True]
    if len(theme_posts) > 0:
        pos = theme_posts.loc[theme_posts[f"{theme} sentiment"] == "Positive", f"{theme} sentiment degree"].sum()
        neg = theme_posts.loc[theme_posts[f"{theme} sentiment"] == "Negative", f"{theme} sentiment degree"].sum()
        total = theme_posts[f"{theme} sentiment degree"].sum()
        return len(theme_posts), (pos-neg)/total
    else:
        return 0, None


print("Number of environmental posts: ", avg_sentiment_calculation("environment")[0], ", Average sentiment of environmental posts: ", avg_sentiment_calculation("environment")[1], sep="")
print("Number of infrastructural posts: ", avg_sentiment_calculation("infrastructure")[0], ", Average sentiment of infrastructural posts: ", avg_sentiment_calculation("infrastructure")[1], sep="")
print("Number of housing-related posts: ", avg_sentiment_calculation("housing")[0], ", Average sentiment of housing-related posts: ", avg_sentiment_calculation("housing")[1], sep="")
print("Number of economic posts: ", avg_sentiment_calculation("economy")[0], ", Average sentiment of economic posts: ", avg_sentiment_calculation("economy")[1], sep="")
print("Number of life-quality-related posts: ", avg_sentiment_calculation("life quality")[0], ", Average sentiment of life-quality-related posts: ", avg_sentiment_calculation("life quality")[1], sep="")
print("Number of aesthetics-related posts: ", avg_sentiment_calculation("aesthetics")[0], ", Average sentiment of aesthetics-related posts: ", avg_sentiment_calculation("aesthetics")[1], sep="")
print("Number of governmental posts: ", avg_sentiment_calculation("government")[0], ", Average sentiment of governmental posts: ", avg_sentiment_calculation("government")[1], sep="")
print("Number of technological posts: ", avg_sentiment_calculation("technology")[0], ", Average sentiment of technological posts: ", avg_sentiment_calculation("technology")[1], sep="")

Number of environmental posts: 0, Average sentiment of environmental posts: None
Number of infrastructural posts: 5, Average sentiment of infrastructural posts: 0.045637871616942154
Number of housing-related posts: 0, Average sentiment of housing-related posts: None
Number of economic posts: 0, Average sentiment of economic posts: None
Number of life-quality-related posts: 0, Average sentiment of life-quality-related posts: None
Number of aesthetics-related posts: 0, Average sentiment of aesthetics-related posts: None
Number of governmental posts: 1, Average sentiment of governmental posts: 1.0
Number of technological posts: 6, Average sentiment of technological posts: 0.866702850429892


In [19]:
posts['year'] = pd.to_datetime(posts['date']).dt.year

year_datasets = {year: posts[posts['year'] == year] for year in range(2010, 2027)}

posts_2010 = year_datasets[2010]
posts_2020 = year_datasets[2020]